In [17]:
import gym
import fh_ac_ai_gym
from itertools import combinations
from collections import defaultdict, deque

WALK = 0
TURNLEFT = 1
TURNRIGHT = 2
GRAB = 3
SHOOT = 4
CLIMB = 5

def Pit(x, y):
    return f"P{x}{y}"

def Wumpus(x, y):
    return f"W{x}{y}"

def Breeze(x, y):
    return f"B{x}{y}"

def Stench(x, y):
    return f"S{x}{y}"

def get_adjacent_cells(x, y):
    adj = []
    if x > 0: adj.append((x - 1, y))
    if x < 3: adj.append((x + 1, y))
    if y > 0: adj.append((x, y - 1))
    if y < 3: adj.append((x, y + 1))
    return adj

wumpus_env = gym.make('Wumpus-v0',disable_env_checker = True)

wumpus_env.reset()
wumpus_env.render()

def Tell(kb,clause) :
    if clause not in kb:
        kb.append(clause)


def negate(literal):
    return literal[1:] if literal.startswith('~') else '~' + literal

def negate_clause(clause):
    return [[negate(lit)] for lit in clause]

def unit_resolution(kb, alpha):
    """
    kb: list of clauses (each clause is a list of literals, e.g., ['~B11', 'P12'])
    alpha: a query clause (e.g., ['P12']) to check entailment
    Returns True if KB ⊨ alpha, i.e., if alpha is entailed by KB
    """
    # Convert KB to frozenset representation
    clauses = set()
    for clause in kb:
        clauses.add(frozenset(clause))
    
    # Negate alpha and add to KB
    neg_alpha = [negate(lit) for lit in alpha]
    for lit in neg_alpha:
        clauses.add(frozenset([lit]))
    
    # Build literal index {literal: set(clauses containing it)}
    index = defaultdict(set)
    for clause in clauses:
        for lit in clause:
            index[lit].add(clause)
    
    unit_queue = deque()
    for clause in clauses:
        if len(clause) == 1:
            unit_queue.append(clause)
    
    seen = set(clauses)
    
    while unit_queue:
        unit_clause = unit_queue.popleft()
        if unit_clause not in clauses:
            continue
        L = next(iter(unit_clause))
        negL = negate(L)
        
        # Check for immediate contradiction
        if frozenset([negL]) in clauses:
            return True
        
        # Process clauses containing L
        for clause in list(index[L]):
            if clause in clauses:
                clauses.remove(clause)
                for lit in clause:
                    index[lit].discard(clause)
        
        # Process clauses containing ~L
        for clause in list(index.get(negL, [])):
            if clause not in clauses:
                continue
            new_clause = clause - {negL}
            
            # Skip tautologies
            if any(negate(lit) in new_clause for lit in new_clause):
                continue
                
            if new_clause in seen:
                continue
                
            seen.add(new_clause)
            
            # Found contradiction
            if not new_clause:
                return True
                
            clauses.add(new_clause)
            for lit in new_clause:
                index[lit].add(new_clause)
            if len(new_clause) == 1:
                unit_queue.append(new_clause)
    
    return False

def resolve(ci, cj):
    resolvents = set()
    for di in ci:
        for dj in cj:
            if di == negate(dj):
                new_clause = (ci - {di}) | (cj - {dj})
                # Filter tautologies early
                if not any(negate(lit) in new_clause for lit in new_clause):
                    resolvents.add(frozenset(new_clause))
    return resolvents

def pl_resolution(kb, alpha):
    clauses = set(frozenset(c) for c in kb)
    neg_alpha = [negate(lit) for lit in alpha]
    
    for lit in neg_alpha:
        clauses.add(frozenset([lit]))
    
    seen = set(clauses)
    
    while True:
        new = set()
        clause_list = list(clauses)
        for (ci, cj) in combinations(clause_list, 2):
            resolvents = resolve(ci, cj)
            for res in resolvents:
                if not res:  # empty clause
                    return True
                if res not in seen:
                    new.add(res)
                    seen.add(res)
        
        if not new:
            return False
        clauses.update(new)


def pl_fc_entails(kb, q):
    count = {}
    inferred = defaultdict(bool)
    agenda = deque()
    # Map each clause to its premises
    premise_map = defaultdict(list)
    
    # Initialize rules and agenda
    for clause in kb:
        premises = clause['premises']
        conclusion = clause['conclusion']
        key = (tuple(premises), conclusion)
        count[key] = len(premises)
        
        for p in premises:
            premise_map[p].append((premises, conclusion))
        
        if not premises:
            agenda.append(conclusion)

    while agenda:
        p = agenda.popleft()
        
        if p == q:
            return True
            
        if not inferred[p]:
            inferred[p] = True
            for (premises, conclusion) in premise_map.get(p, []):
                key = (tuple(premises), conclusion)
                count[key] -= 1
                if count[key] == 0 and not inferred[conclusion]:
                    agenda.append(conclusion)
                        
    return False


def perceive_and_tell(kb, percept, x, y, horn=False):
    # Always assert current cell is safe
    if horn:
        Fact(kb, f"~{Pit(x, y)}")
        Fact(kb, f"~{Wumpus(x, y)}")
    
        # Handle breeze perception
        if percept['breeze']:
            Fact(kb, Breeze(x, y))
        else:
            # No breeze means no pit in adjacent cells
            for (i, j) in get_adjacent_cells(x, y):
                Fact(kb, f"~{Pit(i, j)}")
        
        # Handle stench perception
        if percept['stench']:
            Fact(kb, Stench(x, y))
        else:
            # No stench means no wumpus in adjacent cells
            for (i, j) in get_adjacent_cells(x, y):
                Fact(kb, f"~{Wumpus(i, j)}")
    else:
        Tell(kb, [f"~{Pit(x, y)}"])
        Tell(kb, [f"~{Wumpus(x, y)}"])
        if percept['breeze']:
            Tell(kb, [Breeze(x, y)])
        else:
            Tell(kb, [f"~{Breeze(x, y)}"])
        if percept['stench']:
            Tell(kb, [Stench(x, y)])
        else:
            Tell(kb, [f"~{Stench(x, y)}"])

def is_safe(kb, x, y):
    no_pit = unit_resolution(kb, [f"~{Pit(x, y)}"])
    no_wumpus = unit_resolution(kb, [f"~{Wumpus(x, y)}"])
    return no_pit and no_wumpus

def print_kb(kb):
    print("Knowledge Base:")
    for clause in kb:
        print(" ∨ ".join(clause))


def populate():
    kb = []
    # At least one wumpus
    at_least_one_wumpus = [f"{Wumpus(x,y)}" for x in range(4) for y in range(4) ]

    Tell(kb,at_least_one_wumpus)

    # At most one wumpus

    cells = [(x, y) for x in range(4) for y in range(4)]

    # Replace at-most-one wumpus with pairwise constraints
    for (x1, y1), (x2, y2) in combinations(cells, 2):
            Tell(kb, [f"~{Wumpus(x1,y1)}", f"~{Wumpus(x2,y2)}"])

    # Breeze Rules

    for x in range(4):
        for y in range(4):
            adj = get_adjacent_cells(x, y)

            # ¬Bxy ∨ P_adj1 ∨ P_adj2 ...
            clause = [f"~{Breeze(x, y)}"] + [Pit(i, j) for (i, j) in adj]
            Tell(kb, clause)

            # For each adjacent: ¬Pij ∨ Bxy
            for (i, j) in adj:
                Tell(kb, [f"~{Pit(i,j)}", Breeze(x, y)])

    # Wumpus Rules same as Breeze but with Stink

    for x in range(4):
        for y in range(4):
            adj = get_adjacent_cells(x, y)

            # ¬Sxy ∨ W_adj1 ∨ W_adj2 ...
            clause = [f"~{Stench(x, y)}"] + [Wumpus(i, j) for (i, j) in adj]
            Tell(kb, clause)

            # For each adjacent: ¬Wij ∨ Sxy
            for (i, j) in adj:
                Tell(kb, [f"~{Wumpus(i,j)}", Stench(x, y)])
    return kb



def print_kb_horn(kb):
    print("Knowledge Base:")
    for clause in kb:
        print(clause)


def populate_horn():
    kb = []
    cells = [(x, y) for x in range(4) for y in range(4)]
    
    # At least one Wumpus must exist
    at_least_one_wumpus = [f"{Wumpus(x,y)}" for x in range(4) for y in range(4)]
    Implication(kb, at_least_one_wumpus, "Wumpus_exists")
    
    # At most one Wumpus
    for (x1, y1), (x2, y2) in combinations(cells, 2):
        Implication(kb, [Wumpus(x1, y1), Wumpus(x2, y2)], "false")
    
    # Breeze Rules
    for x in range(4):
        for y in range(4):
            adj = get_adjacent_cells(x, y)
            
            # Pit in adjacent => Breeze
            for (i, j) in adj:
                Implication(kb, [Pit(i, j)], Breeze(x, y))
            
            # Breeze implies at least one pit in adjacent cells
            pit_options = [Pit(i, j) for (i, j) in adj]
            Implication(kb, [Breeze(x, y)], "Pit_near_" + str(x) + str(y))
            Implication(kb, ["Pit_near_" + str(x) + str(y)] + [f"~{Pit(i, j)}" for (i, j) in adj if (i, j) != adj[0]], Pit(adj[0][0], adj[0][1]))
    
    # Stench Rules
    for x in range(4):
        for y in range(4):
            adj = get_adjacent_cells(x, y)
            
            # Wumpus in adjacent => Stench
            for (i, j) in adj:
                Implication(kb, [Wumpus(i, j)], Stench(x, y))
            
            # Stench implies at least one wumpus in adjacent cells
            wumpus_options = [Wumpus(i, j) for (i, j) in adj]
            Implication(kb, [Stench(x, y)], "Wumpus_near_" + str(x) + str(y))
            Implication(kb, ["Wumpus_near_" + str(x) + str(y)] + [f"~{Wumpus(i, j)}" for (i, j) in adj if (i, j) != adj[0]], Wumpus(adj[0][0], adj[0][1]))
    
    return kb

def Fact(kb,literal):
        kb.append({'premises': [], 'conclusion': literal})

def Implication(kb,premises, conclusion):
        if isinstance(premises, str):
            premises = [premises]
        kb.append({'premises': premises, 'conclusion': conclusion})

# Task 3.3 test
kb = populate()

x, y = 0, 0
percept = wumpus_env.reset()
print(percept)
perceive_and_tell(kb,percept, x, y)
for i in range(4):
    for j in range(4):
        if unit_resolution(kb, [Wumpus(i,j)]):
            print(f"Wumpus FOUND at ({i},{j})")


# Move up
print("One up")

wumpus_env.step(TURNLEFT)
next_state, reward, done, info = wumpus_env.step(WALK)
wumpus_env.render()
print(next_state)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
for i in range(4):
    for j in range(4):
        if unit_resolution(kb, [Wumpus(i,j)]):
            print(f"Wumpus FOUND at ({i},{j})")


#move to the left of the pint
percept = wumpus_env.reset()
# reset kb 
kb = populate()

wumpus_env.step(TURNLEFT)
next_state, reward, done, info = wumpus_env.step(WALK)
next_state, reward, done, info = wumpus_env.step(WALK)
wumpus_env.render()
print(next_state)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
found = False

for i in range(4):
    for j in range(4):
        if unit_resolution(kb, [Pit(i,j)]):
            found = True
            print(f"Pit FOUND at ({i},{j})")
if found is False:
    print("Pit was not Found")

## the agent dosent need to make a full circle the pit to know where is 
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
next_state, reward, done, info = wumpus_env.step(TURNRIGHT)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
## its enoought if it goes up right and right couse it it feels BREZE on 0,2 not on 0,3 and again on 1,3 this should be enought to deduct that the pit is on 1,2
## but for some reason to be sure the agents needs to go one more time right
next_state, reward, done, info = wumpus_env.step(TURNRIGHT)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
next_state, reward, done, info = wumpus_env.step(TURNRIGHT)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
next_state, reward, done, info = wumpus_env.step(TURNRIGHT)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'])
wumpus_env.render()


found = False
for i in range(4):
    for j in range(4):
        if unit_resolution(kb, [Pit(i,j)]):
            found = True
            print(f"Pit FOUND at ({i},{j})")
if found is False:
    print("Pit was not Found")

## HORN
kb = populate_horn()

x, y = 0, 0
Fact(kb, f"~{Wumpus(x, y)}")
Fact(kb, f"~{Wumpus(x, y)}")
percept = wumpus_env.reset()
print(percept)
perceive_and_tell(kb,percept, percept['x'], percept['y'],horn=True)
for i in range(4):
    for j in range(4):
        if pl_fc_entails(kb, Wumpus(i,j)):
            print(f"Wumpus FOUND at ({i},{j})")


# Move up
print("One up")

wumpus_env.step(TURNLEFT)
next_state, reward, done, info = wumpus_env.step(WALK)
wumpus_env.render()
print(next_state)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
for i in range(4):
    for j in range(4):
        if pl_fc_entails(kb, Wumpus(i,j)):
            print(f"Wumpus FOUND at ({i},{j})")

#move to the left of the pint
percept = wumpus_env.reset()
# reset kb 
kb = populate_horn()

wumpus_env.step(TURNLEFT)
next_state, reward, done, info = wumpus_env.step(WALK)
next_state, reward, done, info = wumpus_env.step(WALK)
wumpus_env.render()
print(next_state)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
found = False

for i in range(4):
    for j in range(4):
        if pl_fc_entails(kb, Pit(i,j)):
            found = True
            print(f"Pit FOUND at ({i},{j})")
if found is False:
    print("Pit was not Found")

## the agent dosent need to make a full circle the pit to know where is 
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
next_state, reward, done, info = wumpus_env.step(TURNRIGHT)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
## its enoought if it goes up right and right couse it it feels BREZE on 0,2 not on 0,3 and again on 1,3 this should be enought to deduct that the pit is on 1,2
## but for some reason to be sure the agents needs to go one more time right
next_state, reward, done, info = wumpus_env.step(TURNRIGHT)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
next_state, reward, done, info = wumpus_env.step(TURNRIGHT)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
next_state, reward, done, info = wumpus_env.step(TURNRIGHT)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
next_state, reward, done, info = wumpus_env.step(WALK)
perceive_and_tell(kb,next_state, next_state['x'], next_state['y'],horn=True)
wumpus_env.render()

found = False
for i in range(4):
    for j in range(4):
        if pl_fc_entails(kb, Pit(i,j)):
            found = True
            print(f"Pit FOUND at ({i},{j})")
if found is False:
    print("Pit was not Found")

+---+---+---+---+
|   |   |   |  G|
|   |   |   |   |
+---+---+---+---+
|   | P |   |   |
|   |   |   |   |
+---+---+---+---+
|   |   |   |   |
|   |   |   |   |
+---+---+---+---+
|   |W  |   |   |
| A>|   |   |   |
+---+---+---+---+
Perception [ St: True, Br: False, G: False, Bu: False, Sc: False ]
Score : 0
{'x': 0, 'y': 0, 'gold': False, 'direction': <Direction.EAST: 1>, 'arrow': True, 'stench': True, 'breeze': False, 'glitter': False, 'bump': False, 'scream': False}
One up
+---+---+---+---+
|   |   |   |  G|
|   |   |   |   |
+---+---+---+---+
|   | P |   |   |
|   |   |   |   |
+---+---+---+---+
|   |   |   |   |
| A^|   |   |   |
+---+---+---+---+
|   |W  |   |   |
|   |   |   |   |
+---+---+---+---+
Perception [ St: False, Br: False, G: False, Bu: False, Sc: False ]
Score : -2
{'x': 0, 'y': 1, 'gold': False, 'direction': <Direction.NORTH: 0>, 'arrow': True, 'stench': False, 'breeze': False, 'glitter': False, 'bump': False, 'scream': False}
Wumpus FOUND at (1,0)
+---+---+---+---+

![alt text](image.png)

![alt text](image-1.png)

![alt text](image-2.png)

![alt text](image-3.png)


KB CNF after circling PIT (Edit to see it fully)

Knowledge Base:
W00 ∨ W01 ∨ W02 ∨ W03 ∨ W10 ∨ W11 ∨ W12 ∨ W13 ∨ W20 ∨ W21 ∨ W22 ∨ W23 ∨ W30 ∨ W31 ∨ W32 ∨ W33
~W00 ∨ ~W01
~W00 ∨ ~W02
~W00 ∨ ~W03
~W00 ∨ ~W10
~W00 ∨ ~W11
~W00 ∨ ~W12
~W00 ∨ ~W13
~W00 ∨ ~W20
~W00 ∨ ~W21
~W00 ∨ ~W22
~W00 ∨ ~W23
~W00 ∨ ~W30
~W00 ∨ ~W31
~W00 ∨ ~W32
~W00 ∨ ~W33
~W01 ∨ ~W02
~W01 ∨ ~W03
~W01 ∨ ~W10
~W01 ∨ ~W11
~W01 ∨ ~W12
~W01 ∨ ~W13
~W01 ∨ ~W20
~W01 ∨ ~W21
~W01 ∨ ~W22
~W01 ∨ ~W23
~W01 ∨ ~W30
~W01 ∨ ~W31
~W01 ∨ ~W32
~W01 ∨ ~W33
~W02 ∨ ~W03
~W02 ∨ ~W10
~W02 ∨ ~W11
~W02 ∨ ~W12
~W02 ∨ ~W13
~W02 ∨ ~W20
~W02 ∨ ~W21
~W02 ∨ ~W22
~W02 ∨ ~W23
~W02 ∨ ~W30
~W02 ∨ ~W31
~W02 ∨ ~W32
~W02 ∨ ~W33
~W03 ∨ ~W10
~W03 ∨ ~W11
~W03 ∨ ~W12
~W03 ∨ ~W13
~W03 ∨ ~W20
~W03 ∨ ~W21
~W03 ∨ ~W22
~W03 ∨ ~W23
~W03 ∨ ~W30
~W03 ∨ ~W31
~W03 ∨ ~W32
~W03 ∨ ~W33
~W10 ∨ ~W11
~W10 ∨ ~W12
~W10 ∨ ~W13
~W10 ∨ ~W20
~W10 ∨ ~W21
~W10 ∨ ~W22
~W10 ∨ ~W23
~W10 ∨ ~W30
~W10 ∨ ~W31
~W10 ∨ ~W32
~W10 ∨ ~W33
~W11 ∨ ~W12
~W11 ∨ ~W13
~W11 ∨ ~W20
~W11 ∨ ~W21
~W11 ∨ ~W22
~W11 ∨ ~W23
~W11 ∨ ~W30
~W11 ∨ ~W31
~W11 ∨ ~W32
~W11 ∨ ~W33
~W12 ∨ ~W13
~W12 ∨ ~W20
~W12 ∨ ~W21
~W12 ∨ ~W22
~W12 ∨ ~W23
~W12 ∨ ~W30
~W12 ∨ ~W31
~W12 ∨ ~W32
~W12 ∨ ~W33
~W13 ∨ ~W20
~W13 ∨ ~W21
~W13 ∨ ~W22
~W13 ∨ ~W23
~W13 ∨ ~W30
~W13 ∨ ~W31
~W13 ∨ ~W32
~W13 ∨ ~W33
~W20 ∨ ~W21
~W20 ∨ ~W22
~W20 ∨ ~W23
~W20 ∨ ~W30
~W20 ∨ ~W31
~W20 ∨ ~W32
~W20 ∨ ~W33
~W21 ∨ ~W22
~W21 ∨ ~W23
~W21 ∨ ~W30
~W21 ∨ ~W31
~W21 ∨ ~W32
~W21 ∨ ~W33
~W22 ∨ ~W23
~W22 ∨ ~W30
~W22 ∨ ~W31
~W22 ∨ ~W32
~W22 ∨ ~W33
~W23 ∨ ~W30
~W23 ∨ ~W31
~W23 ∨ ~W32
~W23 ∨ ~W33
~W30 ∨ ~W31
~W30 ∨ ~W32
~W30 ∨ ~W33
~W31 ∨ ~W32
~W31 ∨ ~W33
~W32 ∨ ~W33
~B00 ∨ P10 ∨ P01
~P10 ∨ B00
~P01 ∨ B00
~B01 ∨ P11 ∨ P00 ∨ P02
~P11 ∨ B01
~P00 ∨ B01
~P02 ∨ B01
~B02 ∨ P12 ∨ P01 ∨ P03
~P12 ∨ B02
~P01 ∨ B02
~P03 ∨ B02
~B03 ∨ P13 ∨ P02
~P13 ∨ B03
~P02 ∨ B03
~B10 ∨ P00 ∨ P20 ∨ P11
~P00 ∨ B10
~P20 ∨ B10
~P11 ∨ B10
~B11 ∨ P01 ∨ P21 ∨ P10 ∨ P12
~P01 ∨ B11
~P21 ∨ B11
~P10 ∨ B11
~P12 ∨ B11
~B12 ∨ P02 ∨ P22 ∨ P11 ∨ P13
~P02 ∨ B12
~P22 ∨ B12
~P11 ∨ B12
~P13 ∨ B12
~B13 ∨ P03 ∨ P23 ∨ P12
~P03 ∨ B13
~P23 ∨ B13
~P12 ∨ B13
~B20 ∨ P10 ∨ P30 ∨ P21
~P10 ∨ B20
~P30 ∨ B20
~P21 ∨ B20
~B21 ∨ P11 ∨ P31 ∨ P20 ∨ P22
~P11 ∨ B21
~P31 ∨ B21
~P20 ∨ B21
~P22 ∨ B21
~B22 ∨ P12 ∨ P32 ∨ P21 ∨ P23
~P12 ∨ B22
~P32 ∨ B22
~P21 ∨ B22
~P23 ∨ B22
~B23 ∨ P13 ∨ P33 ∨ P22
~P13 ∨ B23
~P33 ∨ B23
~P22 ∨ B23
~B30 ∨ P20 ∨ P31
~P20 ∨ B30
~P31 ∨ B30
~B31 ∨ P21 ∨ P30 ∨ P32
~P21 ∨ B31
~P30 ∨ B31
~P32 ∨ B31
~B32 ∨ P22 ∨ P31 ∨ P33
~P22 ∨ B32
~P31 ∨ B32
~P33 ∨ B32
~B33 ∨ P23 ∨ P32
~P23 ∨ B33
~P32 ∨ B33
~S00 ∨ W10 ∨ W01
~W10 ∨ S00
~W01 ∨ S00
~S01 ∨ W11 ∨ W00 ∨ W02
~W11 ∨ S01
~W00 ∨ S01
~W02 ∨ S01
~S02 ∨ W12 ∨ W01 ∨ W03
~W12 ∨ S02
~W01 ∨ S02
~W03 ∨ S02
~S03 ∨ W13 ∨ W02
~W13 ∨ S03
~W02 ∨ S03
~S10 ∨ W00 ∨ W20 ∨ W11
~W00 ∨ S10
~W20 ∨ S10
~W11 ∨ S10
~S11 ∨ W01 ∨ W21 ∨ W10 ∨ W12
~W01 ∨ S11
~W21 ∨ S11
~W10 ∨ S11
~W12 ∨ S11
~S12 ∨ W02 ∨ W22 ∨ W11 ∨ W13
~W02 ∨ S12
~W22 ∨ S12
~W11 ∨ S12
~W13 ∨ S12
~S13 ∨ W03 ∨ W23 ∨ W12
~W03 ∨ S13
~W23 ∨ S13
~W12 ∨ S13
~S20 ∨ W10 ∨ W30 ∨ W21
~W10 ∨ S20
~W30 ∨ S20
~W21 ∨ S20
~S21 ∨ W11 ∨ W31 ∨ W20 ∨ W22
~W11 ∨ S21
~W31 ∨ S21
~W20 ∨ S21
~W22 ∨ S21
~S22 ∨ W12 ∨ W32 ∨ W21 ∨ W23
~W12 ∨ S22
~W32 ∨ S22
~W21 ∨ S22
~W23 ∨ S22
~S23 ∨ W13 ∨ W33 ∨ W22
~W13 ∨ S23
~W33 ∨ S23
~W22 ∨ S23
~S30 ∨ W20 ∨ W31
~W20 ∨ S30
~W31 ∨ S30
~S31 ∨ W21 ∨ W30 ∨ W32
~W21 ∨ S31
~W30 ∨ S31
~W32 ∨ S31
~S32 ∨ W22 ∨ W31 ∨ W33
~W22 ∨ S32
~W31 ∨ S32
~W33 ∨ S32
~S33 ∨ W23 ∨ W32
~W23 ∨ S33
~W32 ∨ S33
~P02
~W02
B02
~S02
~P03
~W03
~B03
~S03
~P13
~W13
B13
~S13
~P23
~W23
~B23
~S23
~P22
~W22
B22
~S22
~P21
~W21
~B21
~S21
~P11
~W11
B11
S11
~P01
~W01
~B01
~S01

KB after Circling PIT HORN CLAUSES

{'premises': ['W00', 'W01', 'W02', 'W03', 'W10', 'W11', 'W12', 'W13', 'W20', 'W21', 'W22', 'W23', 'W30', 'W31', 'W32', 'W33'], 'conclusion': 'Wumpus_exists'}
{'premises': ['W00', 'W01'], 'conclusion': 'false'}
{'premises': ['W00', 'W02'], 'conclusion': 'false'}
{'premises': ['W00', 'W03'], 'conclusion': 'false'}
{'premises': ['W00', 'W10'], 'conclusion': 'false'}
{'premises': ['W00', 'W11'], 'conclusion': 'false'}
{'premises': ['W00', 'W12'], 'conclusion': 'false'}
{'premises': ['W00', 'W13'], 'conclusion': 'false'}
{'premises': ['W00', 'W20'], 'conclusion': 'false'}
{'premises': ['W00', 'W21'], 'conclusion': 'false'}
{'premises': ['W00', 'W22'], 'conclusion': 'false'}
{'premises': ['W00', 'W23'], 'conclusion': 'false'}
{'premises': ['W00', 'W30'], 'conclusion': 'false'}
{'premises': ['W00', 'W31'], 'conclusion': 'false'}
{'premises': ['W00', 'W32'], 'conclusion': 'false'}
{'premises': ['W00', 'W33'], 'conclusion': 'false'}
{'premises': ['W01', 'W02'], 'conclusion': 'false'}
{'premises': ['W01', 'W03'], 'conclusion': 'false'}
{'premises': ['W01', 'W10'], 'conclusion': 'false'}
{'premises': ['W01', 'W11'], 'conclusion': 'false'}
{'premises': ['W01', 'W12'], 'conclusion': 'false'}
{'premises': ['W01', 'W13'], 'conclusion': 'false'}
{'premises': ['W01', 'W20'], 'conclusion': 'false'}
{'premises': ['W01', 'W21'], 'conclusion': 'false'}
{'premises': ['W01', 'W22'], 'conclusion': 'false'}
{'premises': ['W01', 'W23'], 'conclusion': 'false'}
{'premises': ['W01', 'W30'], 'conclusion': 'false'}
{'premises': ['W01', 'W31'], 'conclusion': 'false'}
{'premises': ['W01', 'W32'], 'conclusion': 'false'}
{'premises': ['W01', 'W33'], 'conclusion': 'false'}
{'premises': ['W02', 'W03'], 'conclusion': 'false'}
{'premises': ['W02', 'W10'], 'conclusion': 'false'}
{'premises': ['W02', 'W11'], 'conclusion': 'false'}
{'premises': ['W02', 'W12'], 'conclusion': 'false'}
{'premises': ['W02', 'W13'], 'conclusion': 'false'}
{'premises': ['W02', 'W20'], 'conclusion': 'false'}
{'premises': ['W02', 'W21'], 'conclusion': 'false'}
{'premises': ['W02', 'W22'], 'conclusion': 'false'}
{'premises': ['W02', 'W23'], 'conclusion': 'false'}
{'premises': ['W02', 'W30'], 'conclusion': 'false'}
{'premises': ['W02', 'W31'], 'conclusion': 'false'}
{'premises': ['W02', 'W32'], 'conclusion': 'false'}
{'premises': ['W02', 'W33'], 'conclusion': 'false'}
{'premises': ['W03', 'W10'], 'conclusion': 'false'}
{'premises': ['W03', 'W11'], 'conclusion': 'false'}
{'premises': ['W03', 'W12'], 'conclusion': 'false'}
{'premises': ['W03', 'W13'], 'conclusion': 'false'}
{'premises': ['W03', 'W20'], 'conclusion': 'false'}
{'premises': ['W03', 'W21'], 'conclusion': 'false'}
{'premises': ['W03', 'W22'], 'conclusion': 'false'}
{'premises': ['W03', 'W23'], 'conclusion': 'false'}
{'premises': ['W03', 'W30'], 'conclusion': 'false'}
{'premises': ['W03', 'W31'], 'conclusion': 'false'}
{'premises': ['W03', 'W32'], 'conclusion': 'false'}
{'premises': ['W03', 'W33'], 'conclusion': 'false'}
{'premises': ['W10', 'W11'], 'conclusion': 'false'}
{'premises': ['W10', 'W12'], 'conclusion': 'false'}
{'premises': ['W10', 'W13'], 'conclusion': 'false'}
{'premises': ['W10', 'W20'], 'conclusion': 'false'}
{'premises': ['W10', 'W21'], 'conclusion': 'false'}
{'premises': ['W10', 'W22'], 'conclusion': 'false'}
{'premises': ['W10', 'W23'], 'conclusion': 'false'}
{'premises': ['W10', 'W30'], 'conclusion': 'false'}
{'premises': ['W10', 'W31'], 'conclusion': 'false'}
{'premises': ['W10', 'W32'], 'conclusion': 'false'}
{'premises': ['W10', 'W33'], 'conclusion': 'false'}
{'premises': ['W11', 'W12'], 'conclusion': 'false'}
{'premises': ['W11', 'W13'], 'conclusion': 'false'}
{'premises': ['W11', 'W20'], 'conclusion': 'false'}
{'premises': ['W11', 'W21'], 'conclusion': 'false'}
{'premises': ['W11', 'W22'], 'conclusion': 'false'}
{'premises': ['W11', 'W23'], 'conclusion': 'false'}
{'premises': ['W11', 'W30'], 'conclusion': 'false'}
{'premises': ['W11', 'W31'], 'conclusion': 'false'}
{'premises': ['W11', 'W32'], 'conclusion': 'false'}
{'premises': ['W11', 'W33'], 'conclusion': 'false'}
{'premises': ['W12', 'W13'], 'conclusion': 'false'}
{'premises': ['W12', 'W20'], 'conclusion': 'false'}
{'premises': ['W12', 'W21'], 'conclusion': 'false'}
{'premises': ['W12', 'W22'], 'conclusion': 'false'}
{'premises': ['W12', 'W23'], 'conclusion': 'false'}
{'premises': ['W12', 'W30'], 'conclusion': 'false'}
{'premises': ['W12', 'W31'], 'conclusion': 'false'}
{'premises': ['W12', 'W32'], 'conclusion': 'false'}
{'premises': ['W12', 'W33'], 'conclusion': 'false'}
{'premises': ['W13', 'W20'], 'conclusion': 'false'}
{'premises': ['W13', 'W21'], 'conclusion': 'false'}
{'premises': ['W13', 'W22'], 'conclusion': 'false'}
{'premises': ['W13', 'W23'], 'conclusion': 'false'}
{'premises': ['W13', 'W30'], 'conclusion': 'false'}
{'premises': ['W13', 'W31'], 'conclusion': 'false'}
{'premises': ['W13', 'W32'], 'conclusion': 'false'}
{'premises': ['W13', 'W33'], 'conclusion': 'false'}
{'premises': ['W20', 'W21'], 'conclusion': 'false'}
{'premises': ['W20', 'W22'], 'conclusion': 'false'}
{'premises': ['W20', 'W23'], 'conclusion': 'false'}
{'premises': ['W20', 'W30'], 'conclusion': 'false'}
{'premises': ['W20', 'W31'], 'conclusion': 'false'}
{'premises': ['W20', 'W32'], 'conclusion': 'false'}
{'premises': ['W20', 'W33'], 'conclusion': 'false'}
{'premises': ['W21', 'W22'], 'conclusion': 'false'}
{'premises': ['W21', 'W23'], 'conclusion': 'false'}
{'premises': ['W21', 'W30'], 'conclusion': 'false'}
{'premises': ['W21', 'W31'], 'conclusion': 'false'}
{'premises': ['W21', 'W32'], 'conclusion': 'false'}
{'premises': ['W21', 'W33'], 'conclusion': 'false'}
{'premises': ['W22', 'W23'], 'conclusion': 'false'}
{'premises': ['W22', 'W30'], 'conclusion': 'false'}
{'premises': ['W22', 'W31'], 'conclusion': 'false'}
{'premises': ['W22', 'W32'], 'conclusion': 'false'}
{'premises': ['W22', 'W33'], 'conclusion': 'false'}
{'premises': ['W23', 'W30'], 'conclusion': 'false'}
{'premises': ['W23', 'W31'], 'conclusion': 'false'}
{'premises': ['W23', 'W32'], 'conclusion': 'false'}
{'premises': ['W23', 'W33'], 'conclusion': 'false'}
{'premises': ['W30', 'W31'], 'conclusion': 'false'}
{'premises': ['W30', 'W32'], 'conclusion': 'false'}
{'premises': ['W30', 'W33'], 'conclusion': 'false'}
{'premises': ['W31', 'W32'], 'conclusion': 'false'}
{'premises': ['W31', 'W33'], 'conclusion': 'false'}
{'premises': ['W32', 'W33'], 'conclusion': 'false'}
{'premises': ['P10'], 'conclusion': 'B00'}
{'premises': ['P01'], 'conclusion': 'B00'}
{'premises': ['B00'], 'conclusion': 'Pit_near_00'}
{'premises': ['Pit_near_00', '~P01'], 'conclusion': 'P10'}
{'premises': ['P11'], 'conclusion': 'B01'}
{'premises': ['P00'], 'conclusion': 'B01'}
{'premises': ['P02'], 'conclusion': 'B01'}
{'premises': ['B01'], 'conclusion': 'Pit_near_01'}
{'premises': ['Pit_near_01', '~P00', '~P02'], 'conclusion': 'P11'}
{'premises': ['P12'], 'conclusion': 'B02'}
{'premises': ['P01'], 'conclusion': 'B02'}
{'premises': ['P03'], 'conclusion': 'B02'}
{'premises': ['B02'], 'conclusion': 'Pit_near_02'}
{'premises': ['Pit_near_02', '~P01', '~P03'], 'conclusion': 'P12'}
{'premises': ['P13'], 'conclusion': 'B03'}
{'premises': ['P02'], 'conclusion': 'B03'}
{'premises': ['B03'], 'conclusion': 'Pit_near_03'}
{'premises': ['Pit_near_03', '~P02'], 'conclusion': 'P13'}
{'premises': ['P00'], 'conclusion': 'B10'}
{'premises': ['P20'], 'conclusion': 'B10'}
{'premises': ['P11'], 'conclusion': 'B10'}
{'premises': ['B10'], 'conclusion': 'Pit_near_10'}
{'premises': ['Pit_near_10', '~P20', '~P11'], 'conclusion': 'P00'}
{'premises': ['P01'], 'conclusion': 'B11'}
{'premises': ['P21'], 'conclusion': 'B11'}
{'premises': ['P10'], 'conclusion': 'B11'}
{'premises': ['P12'], 'conclusion': 'B11'}
{'premises': ['B11'], 'conclusion': 'Pit_near_11'}
{'premises': ['Pit_near_11', '~P21', '~P10', '~P12'], 'conclusion': 'P01'}
{'premises': ['P02'], 'conclusion': 'B12'}
{'premises': ['P22'], 'conclusion': 'B12'}
{'premises': ['P11'], 'conclusion': 'B12'}
{'premises': ['P13'], 'conclusion': 'B12'}
{'premises': ['B12'], 'conclusion': 'Pit_near_12'}
{'premises': ['Pit_near_12', '~P22', '~P11', '~P13'], 'conclusion': 'P02'}
{'premises': ['P03'], 'conclusion': 'B13'}
{'premises': ['P23'], 'conclusion': 'B13'}
{'premises': ['P12'], 'conclusion': 'B13'}
{'premises': ['B13'], 'conclusion': 'Pit_near_13'}
{'premises': ['Pit_near_13', '~P23', '~P12'], 'conclusion': 'P03'}
{'premises': ['P10'], 'conclusion': 'B20'}
{'premises': ['P30'], 'conclusion': 'B20'}
{'premises': ['P21'], 'conclusion': 'B20'}
{'premises': ['B20'], 'conclusion': 'Pit_near_20'}
{'premises': ['Pit_near_20', '~P30', '~P21'], 'conclusion': 'P10'}
{'premises': ['P11'], 'conclusion': 'B21'}
{'premises': ['P31'], 'conclusion': 'B21'}
{'premises': ['P20'], 'conclusion': 'B21'}
{'premises': ['P22'], 'conclusion': 'B21'}
{'premises': ['B21'], 'conclusion': 'Pit_near_21'}
{'premises': ['Pit_near_21', '~P31', '~P20', '~P22'], 'conclusion': 'P11'}
{'premises': ['P12'], 'conclusion': 'B22'}
{'premises': ['P32'], 'conclusion': 'B22'}
{'premises': ['P21'], 'conclusion': 'B22'}
{'premises': ['P23'], 'conclusion': 'B22'}
{'premises': ['B22'], 'conclusion': 'Pit_near_22'}
{'premises': ['Pit_near_22', '~P32', '~P21', '~P23'], 'conclusion': 'P12'}
{'premises': ['P13'], 'conclusion': 'B23'}
{'premises': ['P33'], 'conclusion': 'B23'}
{'premises': ['P22'], 'conclusion': 'B23'}
{'premises': ['B23'], 'conclusion': 'Pit_near_23'}
{'premises': ['Pit_near_23', '~P33', '~P22'], 'conclusion': 'P13'}
{'premises': ['P20'], 'conclusion': 'B30'}
{'premises': ['P31'], 'conclusion': 'B30'}
{'premises': ['B30'], 'conclusion': 'Pit_near_30'}
{'premises': ['Pit_near_30', '~P31'], 'conclusion': 'P20'}
{'premises': ['P21'], 'conclusion': 'B31'}
{'premises': ['P30'], 'conclusion': 'B31'}
{'premises': ['P32'], 'conclusion': 'B31'}
{'premises': ['B31'], 'conclusion': 'Pit_near_31'}
{'premises': ['Pit_near_31', '~P30', '~P32'], 'conclusion': 'P21'}
{'premises': ['P22'], 'conclusion': 'B32'}
{'premises': ['P31'], 'conclusion': 'B32'}
{'premises': ['P33'], 'conclusion': 'B32'}
{'premises': ['B32'], 'conclusion': 'Pit_near_32'}
{'premises': ['Pit_near_32', '~P31', '~P33'], 'conclusion': 'P22'}
{'premises': ['P23'], 'conclusion': 'B33'}
{'premises': ['P32'], 'conclusion': 'B33'}
{'premises': ['B33'], 'conclusion': 'Pit_near_33'}
{'premises': ['Pit_near_33', '~P32'], 'conclusion': 'P23'}
{'premises': ['W10'], 'conclusion': 'S00'}
{'premises': ['W01'], 'conclusion': 'S00'}
{'premises': ['S00'], 'conclusion': 'Wumpus_near_00'}
{'premises': ['Wumpus_near_00', '~W01'], 'conclusion': 'W10'}
{'premises': ['W11'], 'conclusion': 'S01'}
{'premises': ['W00'], 'conclusion': 'S01'}
{'premises': ['W02'], 'conclusion': 'S01'}
{'premises': ['S01'], 'conclusion': 'Wumpus_near_01'}
{'premises': ['Wumpus_near_01', '~W00', '~W02'], 'conclusion': 'W11'}
{'premises': ['W12'], 'conclusion': 'S02'}
{'premises': ['W01'], 'conclusion': 'S02'}
{'premises': ['W03'], 'conclusion': 'S02'}
{'premises': ['S02'], 'conclusion': 'Wumpus_near_02'}
{'premises': ['Wumpus_near_02', '~W01', '~W03'], 'conclusion': 'W12'}
{'premises': ['W13'], 'conclusion': 'S03'}
{'premises': ['W02'], 'conclusion': 'S03'}
{'premises': ['S03'], 'conclusion': 'Wumpus_near_03'}
{'premises': ['Wumpus_near_03', '~W02'], 'conclusion': 'W13'}
{'premises': ['W00'], 'conclusion': 'S10'}
{'premises': ['W20'], 'conclusion': 'S10'}
{'premises': ['W11'], 'conclusion': 'S10'}
{'premises': ['S10'], 'conclusion': 'Wumpus_near_10'}
{'premises': ['Wumpus_near_10', '~W20', '~W11'], 'conclusion': 'W00'}
{'premises': ['W01'], 'conclusion': 'S11'}
{'premises': ['W21'], 'conclusion': 'S11'}
{'premises': ['W10'], 'conclusion': 'S11'}
{'premises': ['W12'], 'conclusion': 'S11'}
{'premises': ['S11'], 'conclusion': 'Wumpus_near_11'}
{'premises': ['Wumpus_near_11', '~W21', '~W10', '~W12'], 'conclusion': 'W01'}
{'premises': ['W02'], 'conclusion': 'S12'}
{'premises': ['W22'], 'conclusion': 'S12'}
{'premises': ['W11'], 'conclusion': 'S12'}
{'premises': ['W13'], 'conclusion': 'S12'}
{'premises': ['S12'], 'conclusion': 'Wumpus_near_12'}
{'premises': ['Wumpus_near_12', '~W22', '~W11', '~W13'], 'conclusion': 'W02'}
{'premises': ['W03'], 'conclusion': 'S13'}
{'premises': ['W23'], 'conclusion': 'S13'}
{'premises': ['W12'], 'conclusion': 'S13'}
{'premises': ['S13'], 'conclusion': 'Wumpus_near_13'}
{'premises': ['Wumpus_near_13', '~W23', '~W12'], 'conclusion': 'W03'}
{'premises': ['W10'], 'conclusion': 'S20'}
{'premises': ['W30'], 'conclusion': 'S20'}
{'premises': ['W21'], 'conclusion': 'S20'}
{'premises': ['S20'], 'conclusion': 'Wumpus_near_20'}
{'premises': ['Wumpus_near_20', '~W30', '~W21'], 'conclusion': 'W10'}
{'premises': ['W11'], 'conclusion': 'S21'}
{'premises': ['W31'], 'conclusion': 'S21'}
{'premises': ['W20'], 'conclusion': 'S21'}
{'premises': ['W22'], 'conclusion': 'S21'}
{'premises': ['S21'], 'conclusion': 'Wumpus_near_21'}
{'premises': ['Wumpus_near_21', '~W31', '~W20', '~W22'], 'conclusion': 'W11'}
{'premises': ['W12'], 'conclusion': 'S22'}
{'premises': ['W32'], 'conclusion': 'S22'}
{'premises': ['W21'], 'conclusion': 'S22'}
{'premises': ['W23'], 'conclusion': 'S22'}
{'premises': ['S22'], 'conclusion': 'Wumpus_near_22'}
{'premises': ['Wumpus_near_22', '~W32', '~W21', '~W23'], 'conclusion': 'W12'}
{'premises': ['W13'], 'conclusion': 'S23'}
{'premises': ['W33'], 'conclusion': 'S23'}
{'premises': ['W22'], 'conclusion': 'S23'}
{'premises': ['S23'], 'conclusion': 'Wumpus_near_23'}
{'premises': ['Wumpus_near_23', '~W33', '~W22'], 'conclusion': 'W13'}
{'premises': ['W20'], 'conclusion': 'S30'}
{'premises': ['W31'], 'conclusion': 'S30'}
{'premises': ['S30'], 'conclusion': 'Wumpus_near_30'}
{'premises': ['Wumpus_near_30', '~W31'], 'conclusion': 'W20'}
{'premises': ['W21'], 'conclusion': 'S31'}
{'premises': ['W30'], 'conclusion': 'S31'}
{'premises': ['W32'], 'conclusion': 'S31'}
{'premises': ['S31'], 'conclusion': 'Wumpus_near_31'}
{'premises': ['Wumpus_near_31', '~W30', '~W32'], 'conclusion': 'W21'}
{'premises': ['W22'], 'conclusion': 'S32'}
{'premises': ['W31'], 'conclusion': 'S32'}
{'premises': ['W33'], 'conclusion': 'S32'}
{'premises': ['S32'], 'conclusion': 'Wumpus_near_32'}
{'premises': ['Wumpus_near_32', '~W31', '~W33'], 'conclusion': 'W22'}
{'premises': ['W23'], 'conclusion': 'S33'}
{'premises': ['W32'], 'conclusion': 'S33'}
{'premises': ['S33'], 'conclusion': 'Wumpus_near_33'}
{'premises': ['Wumpus_near_33', '~W32'], 'conclusion': 'W23'}
{'premises': [], 'conclusion': '~P02'}
{'premises': [], 'conclusion': '~W02'}
{'premises': [], 'conclusion': 'B02'}
{'premises': [], 'conclusion': '~W12'}
{'premises': [], 'conclusion': '~W01'}
{'premises': [], 'conclusion': '~W03'}
{'premises': [], 'conclusion': '~P03'}
{'premises': [], 'conclusion': '~W03'}
{'premises': [], 'conclusion': '~P13'}
{'premises': [], 'conclusion': '~P02'}
{'premises': [], 'conclusion': '~W13'}
{'premises': [], 'conclusion': '~W02'}
{'premises': [], 'conclusion': '~P03'}
{'premises': [], 'conclusion': '~W03'}
{'premises': [], 'conclusion': '~P13'}
{'premises': [], 'conclusion': '~P02'}
{'premises': [], 'conclusion': '~W13'}
{'premises': [], 'conclusion': '~W02'}
{'premises': [], 'conclusion': '~P13'}
{'premises': [], 'conclusion': '~W13'}
{'premises': [], 'conclusion': 'B13'}
{'premises': [], 'conclusion': '~W03'}
{'premises': [], 'conclusion': '~W23'}
{'premises': [], 'conclusion': '~W12'}
{'premises': [], 'conclusion': '~P23'}
{'premises': [], 'conclusion': '~W23'}
{'premises': [], 'conclusion': '~P13'}
{'premises': [], 'conclusion': '~P33'}
{'premises': [], 'conclusion': '~P22'}
{'premises': [], 'conclusion': '~W13'}
{'premises': [], 'conclusion': '~W33'}
{'premises': [], 'conclusion': '~W22'}
{'premises': [], 'conclusion': '~P23'}
{'premises': [], 'conclusion': '~W23'}
{'premises': [], 'conclusion': '~P13'}
{'premises': [], 'conclusion': '~P33'}
{'premises': [], 'conclusion': '~P22'}
{'premises': [], 'conclusion': '~W13'}
{'premises': [], 'conclusion': '~W33'}
{'premises': [], 'conclusion': '~W22'}
{'premises': [], 'conclusion': '~P22'}
{'premises': [], 'conclusion': '~W22'}
{'premises': [], 'conclusion': 'B22'}
{'premises': [], 'conclusion': '~W12'}
{'premises': [], 'conclusion': '~W32'}
{'premises': [], 'conclusion': '~W21'}
{'premises': [], 'conclusion': '~W23'}
{'premises': [], 'conclusion': '~P21'}
{'premises': [], 'conclusion': '~W21'}
{'premises': [], 'conclusion': '~P11'}
{'premises': [], 'conclusion': '~P31'}
{'premises': [], 'conclusion': '~P20'}
{'premises': [], 'conclusion': '~P22'}
{'premises': [], 'conclusion': '~W11'}
{'premises': [], 'conclusion': '~W31'}
{'premises': [], 'conclusion': '~W20'}
{'premises': [], 'conclusion': '~W22'}
{'premises': [], 'conclusion': '~P21'}
{'premises': [], 'conclusion': '~W21'}
{'premises': [], 'conclusion': '~P11'}
{'premises': [], 'conclusion': '~P31'}
{'premises': [], 'conclusion': '~P20'}
{'premises': [], 'conclusion': '~P22'}
{'premises': [], 'conclusion': '~W11'}
{'premises': [], 'conclusion': '~W31'}
{'premises': [], 'conclusion': '~W20'}
{'premises': [], 'conclusion': '~W22'}
{'premises': [], 'conclusion': '~P11'}
{'premises': [], 'conclusion': '~W11'}
{'premises': [], 'conclusion': 'B11'}
{'premises': [], 'conclusion': 'S11'}
{'premises': [], 'conclusion': '~P01'}
{'premises': [], 'conclusion': '~W01'}
{'premises': [], 'conclusion': '~P11'}
{'premises': [], 'conclusion': '~P00'}
{'premises': [], 'conclusion': '~P02'}
{'premises': [], 'conclusion': '~W11'}
{'premises': [], 'conclusion': '~W00'}
{'premises': [], 'conclusion': '~W02'}
{'premises': [], 'conclusion': '~P01'}
{'premises': [], 'conclusion': '~W01'}
{'premises': [], 'conclusion': '~P11'}
{'premises': [], 'conclusion': '~P00'}
{'premises': [], 'conclusion': '~P02'}
{'premises': [], 'conclusion': '~W11'}
{'premises': [], 'conclusion': '~W00'}
{'premises': [], 'conclusion': '~W02'}
{'premises': [], 'conclusion': '~P02'}
{'premises': [], 'conclusion': '~W02'}
{'premises': [], 'conclusion': 'B02'}
{'premises': [], 'conclusion': '~W12'}
{'premises': [], 'conclusion': '~W01'}
{'premises': [], 'conclusion': '~W03'}